# LeetCode 1177 — Can Make Palindrome from Substring bằng Python

Link đề: https://leetcode.com/problems/can-make-palindrome-from-substring/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Bit Manipulation, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def canMakePaliQueries(self, s: str, queries: list[list[int]]) -> list[bool]:
        ...
```

Ghi chú: đáp án là **mảng boolean**, mỗi truy vấn `[left, right, k]` hỏi: sau khi **sắp xếp lại tuỳ ý** đoạn `s[left..right]` và **thay tối đa `k` ký tự**, có thể thành palindrome không? Điểm cốt lõi: chỉ **tính chẵn lẻ** của số lần xuất hiện mỗi chữ cái là quan trọng; gọi `odd` = số chữ cái xuất hiện lẻ lần, câu trả lời là `odd // 2 <= k`. Trả lời `10^5` truy vấn nhanh bằng **tổng tiền tố** hoặc **bitmask XOR tiền tố**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: được **hoán vị** đoạn con rồi **thay** tối đa `k` ký tự; mỗi truy vấn độc lập, không sửa `s`.
2. Nắm điều kiện palindrome theo tần suất: một multiset ký tự xếp được thành palindrome ⟺ có **tối đa một** ký tự xuất hiện lẻ lần; mỗi lần thay giảm số ký tự lẻ đi **2**.
3. Nắm công cụ: `Counter`, tổng tiền tố 2 chiều `pre[i][c]`, **bitmask** 26 bit, phép **XOR** (`^`), đếm bit 1 bằng `int.bit_count()`.
4. Đi từ **đếm lại từng đoạn** → **tổng tiền tố 26 cột** → **tiền tố chẵn lẻ** → **bitmask XOR tiền tố**.
5. Tránh lỗi: dùng `odd <= k` thay vì `odd // 2 <= k`, lệch chỉ số khi dùng tiền tố (`right + 1`), quên đoạn độ dài lẻ được phép một ký tự lẻ.

## 2. Đề bài & ví dụ

Cho chuỗi `s` và mảng `queries`, `queries[i] = [left_i, right_i, k_i]`. Với mỗi truy vấn, ta được **sắp xếp lại** đoạn `s[left_i..right_i]` rồi chọn **tối đa `k_i`** ký tự trong đoạn để **thay** bằng chữ thường bất kỳ.

Nếu đoạn có thể trở thành **palindrome** sau các thao tác trên thì kết quả là `true`, ngược lại `false`. Trả về mảng `answer`.

Lưu ý: mỗi ký tự được đếm riêng khi thay (ví dụ đoạn `"aaa"` với `k = 2` chỉ thay được 2 ký tự). Truy vấn **không** sửa `s`.

**Ví dụ 1:** `s = "abcda"`, `queries = [[3,3,0],[1,2,0],[0,3,1],[0,3,2],[0,4,1]]` → `[true,false,false,true,true]`.

- `"d"` đã là palindrome; `"bc"` không; `"abcd"` thay 1 không đủ; `"abcd"` thay 2 → `"abba"`; `"abcda"` thay 1 → `"abcba"`.

**Ví dụ 2:** `s = "lyb"`, `queries = [[0,1,0],[2,2,1]]` → `[false,true]`.

**Ràng buộc:** `1 <= s.length, queries.length <= 10^5`; `0 <= left <= right < s.length`; `0 <= k <= s.length`; `s` chỉ gồm chữ thường.

In [1]:
import random
import time
from collections import Counter
from itertools import permutations, product
from typing import List

examples = [
    ("abcda", [[3, 3, 0], [1, 2, 0], [0, 3, 1], [0, 3, 2], [0, 4, 1]], [True, False, False, True, True]),
    ("lyb", [[0, 1, 0], [2, 2, 1]], [False, True]),
]
for s, q, want in examples:
    print(s, q, "->", want)

abcda [[3, 3, 0], [1, 2, 0], [0, 3, 1], [0, 3, 2], [0, 4, 1]] -> [True, False, False, True, True]
lyb [[0, 1, 0], [2, 2, 1]] -> [False, True]


## 3. Kiến thức nền

### 3.1 Palindrome và điều kiện theo tần suất

**Palindrome** là chuỗi đọc xuôi ngược như nhau. Khi được **hoán vị tự do**, chỉ số lần xuất hiện quan trọng: các ký tự ghép cặp đối xứng hai bên, chỉ **một** ký tự (ở giữa, khi độ dài lẻ) được phép không có cặp. Vậy: xếp được palindrome ⟺ `odd ≤ 1`, với `odd` = số chữ cái có tần suất lẻ.

In [2]:
def odd_count(t):
    return sum(v % 2 for v in Counter(t).values())

for t in ["aabb", "aab", "abc", "abcd", "abcda"]:
    print(f"{t!r:8} odd={odd_count(t)} xếp được palindrome? {odd_count(t) <= 1}")

'aabb'   odd=0 xếp được palindrome? True
'aab'    odd=1 xếp được palindrome? True
'abc'    odd=3 xếp được palindrome? False
'abcd'   odd=4 xếp được palindrome? False
'abcda'  odd=3 xếp được palindrome? False


### 3.2 Một lần thay giảm `odd` đi 2

Thay một ký tự `x` (đang lẻ) bằng ký tự `y` (đang lẻ): `x` thành chẵn, `y` thành chẵn → `odd` giảm 2. Đó là cách dùng một lần thay hiệu quả nhất. Từ `odd` cần đưa về `≤ 1` → cần `odd // 2` lần thay. Điều kiện: **`odd // 2 <= k`**.

Kiểm chứng bằng brute force thật sự (thử mọi cách thay) trên chuỗi rất ngắn:

In [3]:
def can_brute_force_tiny(t, k, alphabet="abcd"):
    """Thử mọi chuỗi cùng độ dài khác t ở tối đa k vị trí (sau khi được hoán vị) — chỉ cho chuỗi rất ngắn."""
    n = len(t)
    for cand in product(alphabet, repeat=n):
        # cand có thể xếp thành palindrome và tạo từ t bằng <= k lần thay (vì được hoán vị, chỉ so multiset)
        if odd_count(cand) <= 1:
            common = sum((Counter(t) & Counter(cand)).values())
            if n - common <= k:
                return True
    return False

for t in ["ab", "abc", "abcd", "aabc"]:
    for k in range(3):
        assert can_brute_force_tiny(t, k) == (odd_count(t) // 2 <= k), (t, k)
print("odd // 2 <= k khớp brute force thử mọi cách thay")

odd // 2 <= k khớp brute force thử mọi cách thay


### 3.3 Tổng tiền tố (prefix sum) theo từng chữ cái

`pre[i][c]` = số lần chữ `c` xuất hiện trong `s[:i]`. Số lần trong `s[l..r]` = `pre[r+1][c] − pre[l][c]`. Tiền xử lý `O(26n)`, mỗi truy vấn `O(26)`.

In [4]:
s0 = "abcda"
pre = [[0] * 26]
for ch in s0:
    row = pre[-1][:]
    row[ord(ch) - 97] += 1
    pre.append(row)
l, r = 0, 3
cnt = [pre[r + 1][c] - pre[l][c] for c in range(26)]
print("đếm 'abcd':", {chr(97 + c): v for c, v in enumerate(cnt) if v})

đếm 'abcd': {'a': 1, 'b': 1, 'c': 1, 'd': 1}


### 3.4 Phép XOR và bitmask

Biểu diễn **tính chẵn lẻ** của 26 chữ cái bằng một số nguyên 26 bit: bit `c` = 1 nếu chữ `c` xuất hiện lẻ lần. Thêm một chữ `c` = **lật** bit `c` = `mask ^= 1 << c`. XOR có tính chất `x ^ x = 0`, nên chẵn lẻ của đoạn `s[l..r]` = `mask[r+1] ^ mask[l]`.

In [5]:
m = 0
for ch in "abca":
    m ^= 1 << (ord(ch) - 97)
print(bin(m), "-> bit b và c bật (a xuất hiện 2 lần nên tắt)")
assert m == (1 << 1) | (1 << 2)

0b110 -> bit b và c bật (a xuất hiện 2 lần nên tắt)


### 3.5 Đếm bit 1: `int.bit_count()`

Python 3.10+ có `x.bit_count()` đếm số bit 1 (popcount). Bản cũ dùng `bin(x).count("1")`. Số bit 1 của mask đoạn chính là `odd`.

In [6]:
x = 0b101101
print(x.bit_count(), bin(x).count("1"))
assert x.bit_count() == 4

4 4


## 4. Ý tưởng nền tảng / trực giác

Tính tay `s = "abcda"`:

| Truy vấn | Đoạn | Tần suất | odd | odd // 2 | k | Kết quả |
|----------|------|----------|-----|----------|---|---------|
| [3,3,0] | d | d:1 | 1 | 0 | 0 | true |
| [1,2,0] | bc | b:1 c:1 | 2 | 1 | 0 | false |
| [0,3,1] | abcd | 4 chữ lẻ | 4 | 2 | 1 | false |
| [0,3,2] | abcd | | 4 | 2 | 2 | true |
| [0,4,1] | abcda | a:2, b,c,d:1 | 3 | 1 | 1 | true |

Toàn bộ thông tin cần cho mỗi truy vấn là **một số `odd`**. Vấn đề còn lại: tính `odd` của `10^5` đoạn nhanh.

In [7]:
for l, r, k in examples[0][1]:
    t = examples[0][0][l:r + 1]
    o = odd_count(t)
    print(f"[{l},{r},{k}] {t!r:8} odd={o} -> {o // 2 <= k}")

[3,3,0] 'd'      odd=1 -> True
[1,2,0] 'bc'     odd=2 -> False
[0,3,1] 'abcd'   odd=4 -> False
[0,3,2] 'abcd'   odd=4 -> True
[0,4,1] 'abcda'  odd=3 -> True


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi truy vấn, cắt đoạn, đếm bằng `Counter`, tính `odd`, so với `k`.

In [8]:
def can_make_brute(s, queries):
    res = []
    for l, r, k in queries:
        odd = sum(v % 2 for v in Counter(s[l:r + 1]).values())
        res.append(odd // 2 <= k)
    return res

for s, q, want in examples:
    got = can_make_brute(s, q)
    print(got)
    assert got == want

[True, False, False, True, True]
[False, True]


**Độ phức tạp:** mỗi truy vấn `O(độ dài đoạn)` → tệ nhất `O(n · Q)` = `10^10` — quá chậm.

**Nút thắt:** đếm lại các ký tự đã đếm ở truy vấn khác. Các đoạn chồng lên nhau rất nhiều.

## 6. Bước đột phá

> Số lần xuất hiện trong đoạn = hiệu hai **tổng tiền tố**. Và vì chỉ cần **chẵn lẻ**, hiệu trở thành **XOR** của hai **mask tiền tố** — một phép toán trên một số nguyên.

Invariant: `mask[i]` = chẵn lẻ của các chữ cái trong `s[:i]`. Khi đó:

- `mask[r+1] ^ mask[l]` = chẵn lẻ trong `s[l..r]` (phần chung `s[:l]` triệt tiêu vì `x ^ x = 0`).
- `odd = (mask[r+1] ^ mask[l]).bit_count()`.

Quy tắc: nếu `odd // 2 <= k` thì `True`, ngược lại `False`.

Mẹo thêm: nếu `k >= 13` thì luôn `True` (vì `odd ≤ 26` ⇒ `odd // 2 ≤ 13`).

In [9]:
s0 = "abcda"
mask = [0]
for ch in s0:
    mask.append(mask[-1] ^ (1 << (ord(ch) - 97)))
print([bin(m) for m in mask])
l, r = 0, 4
print("mask đoạn abcda:", bin(mask[r + 1] ^ mask[l]), "odd =", (mask[r + 1] ^ mask[l]).bit_count())

['0b0', '0b1', '0b11', '0b111', '0b1111', '0b1110']
mask đoạn abcda: 0b1110 odd = 3


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tổng tiền tố 26 cột

In [10]:
def can_make_prefix_counts(s, queries):
    n = len(s)
    pre = [[0] * 26 for _ in range(n + 1)]
    for i, ch in enumerate(s):
        pre[i + 1] = pre[i][:]
        pre[i + 1][ord(ch) - 97] += 1
    res = []
    for l, r, k in queries:
        a, b = pre[r + 1], pre[l]
        odd = sum((a[c] - b[c]) & 1 for c in range(26))
        res.append(odd // 2 <= k)
    return res

for s, q, want in examples:
    assert can_make_prefix_counts(s, q) == want
print("prefix counts OK")

prefix counts OK


**Độ phức tạp:** tiền xử lý `O(26n)` thời gian và bộ nhớ (`26 · 10^5` số nguyên ~ vài chục MB trong Python — nặng), mỗi truy vấn `O(26)`.

### 7.2 Cách 3 — Tiền tố chẵn lẻ (mảng 0/1)

Chỉ lưu `par[i][c] = pre[i][c] % 2`. Chẵn lẻ của đoạn: `par[r+1][c] != par[l][c]`. Cùng độ phức tạp nhưng là bước chuyển tư duy sang bitmask.

In [11]:
def can_make_prefix_parity(s, queries):
    par = [[0] * 26]
    for ch in s:
        row = par[-1][:]
        row[ord(ch) - 97] ^= 1
        par.append(row)
    res = []
    for l, r, k in queries:
        a, b = par[r + 1], par[l]
        odd = sum(a[c] != b[c] for c in range(26))
        res.append(odd // 2 <= k)
    return res

for s, q, want in examples:
    assert can_make_prefix_parity(s, q) == want
print("prefix parity OK")

prefix parity OK


### 7.3 Cách 4 — Bitmask XOR tiền tố (lời giải chuẩn)

Nén 26 bit chẵn lẻ vào **một** số nguyên. Tiền xử lý `O(n)`, mỗi truy vấn `O(1)` (XOR + popcount).

In [12]:
def can_make_bitmask(s, queries):
    mask = [0] * (len(s) + 1)
    for i, ch in enumerate(s):
        mask[i + 1] = mask[i] ^ (1 << (ord(ch) - 97))
    return [(mask[r + 1] ^ mask[l]).bit_count() // 2 <= k for l, r, k in queries]

for s, q, want in examples:
    assert can_make_bitmask(s, q) == want
print("bitmask OK")

bitmask OK


**Độ phức tạp:** `O(n + Q)` thời gian, `O(n)` bộ nhớ.

### 7.4 Cách 5 — Bitmask + cắt sớm `k >= 13` và đoạn ngắn

Tối ưu nhỏ thực tế: nếu `k >= 13` hoặc `2k + 1 >= độ dài đoạn` (có thể thay nửa đoạn để ghép đối xứng) thì chắc chắn `True`, không cần tính mask. Không đổi độ phức tạp, minh hoạ cách lập luận cận.

In [13]:
def can_make_bitmask_early(s, queries):
    mask = [0] * (len(s) + 1)
    for i, ch in enumerate(s):
        mask[i + 1] = mask[i] ^ (1 << (ord(ch) - 97))
    res = []
    for l, r, k in queries:
        if k >= 13 or 2 * k + 1 >= r - l + 1:
            res.append(True)
        else:
            res.append((mask[r + 1] ^ mask[l]).bit_count() // 2 <= k)
    return res

for s, q, want in examples:
    assert can_make_bitmask_early(s, q) == want
print("bitmask early OK")

bitmask early OK


Vì sao `2k + 1 >= len` đủ? Đoạn độ dài `L` có tối đa `L` ký tự lẻ (khi mọi ký tự khác nhau, `odd = L`); hơn nữa `odd ≡ L (mod 2)`, nên `odd // 2 ≤ L // 2`. Nếu `k ≥ L // 2` (tương đương `2k + 1 ≥ L`) thì luôn đủ.

## 8. TRACE từng bước

Trace cách 4 trên `s = "abcda"`:

| i | s[i-1] | mask[i] (bit e..a) |
|---|--------|--------------------|
| 0 | | 00000 |
| 1 | a | 00001 |
| 2 | b | 00011 |
| 3 | c | 00111 |
| 4 | d | 01111 |
| 5 | a | 01110 |

Truy vấn `[0,4,1]`: `mask[5] ^ mask[0] = 01110` → 3 bit → `3 // 2 = 1 ≤ 1` → `True`.

In [14]:
def trace_bitmask(s, queries):
    mask = [0] * (len(s) + 1)
    for i, ch in enumerate(s):
        mask[i + 1] = mask[i] ^ (1 << (ord(ch) - 97))
        print(f"i={i + 1} thêm {ch!r} -> mask={mask[i + 1]:05b}")
    res = []
    for l, r, k in queries:
        m = mask[r + 1] ^ mask[l]
        odd = m.bit_count()
        ok = odd // 2 <= k
        res.append(ok)
        print(f"query [{l},{r},{k}] mask[{r + 1}]^mask[{l}] = {mask[r + 1]:05b}^{mask[l]:05b} = {m:05b}  odd={odd} -> {ok}")
    return res

assert trace_bitmask(*examples[0][:2]) == examples[0][2]

i=1 thêm 'a' -> mask=00001
i=2 thêm 'b' -> mask=00011
i=3 thêm 'c' -> mask=00111
i=4 thêm 'd' -> mask=01111
i=5 thêm 'a' -> mask=01110
query [3,3,0] mask[4]^mask[3] = 01111^00111 = 01000  odd=1 -> True
query [1,2,0] mask[3]^mask[1] = 00111^00001 = 00110  odd=2 -> False
query [0,3,1] mask[4]^mask[0] = 01111^00000 = 01111  odd=4 -> False
query [0,3,2] mask[4]^mask[0] = 01111^00000 = 01111  odd=4 -> True
query [0,4,1] mask[5]^mask[0] = 01110^00000 = 01110  odd=3 -> True


## 9. Code nộp LeetCode

In [15]:
from typing import List


class Solution:
    def canMakePaliQueries(self, s: str, queries: List[List[int]]) -> List[bool]:
        mask = [0] * (len(s) + 1)
        for i, ch in enumerate(s):
            mask[i + 1] = mask[i] ^ (1 << (ord(ch) - 97))   # lật bit chẵn lẻ của ch
        res = []
        for l, r, k in queries:
            odd = bin(mask[r + 1] ^ mask[l]).count("1")      # số chữ xuất hiện lẻ lần trong s[l..r]
            res.append(odd // 2 <= k)
        return res


sol = Solution()
for s, q, want in examples:
    assert sol.canMakePaliQueries(s, q) == want
print("Solution OK")

Solution OK


(Dùng `bin(x).count("1")` để chạy được cả trên phiên bản Python cũ; trên 3.10+ có thể dùng `x.bit_count()`.)

## 10. Vì sao đúng

**Bổ đề 1.** Multiset ký tự độ dài `L` xếp được thành palindrome ⟺ `odd ≤ 1`. (⇐) ghép từng cặp ký tự giống nhau vào hai vị trí đối xứng, ký tự lẻ (nếu có) đặt giữa. (⇒) trong palindrome, mọi ký tự ngoài vị trí giữa đều có cặp đối xứng giống nó.

**Bổ đề 2.** Số lần thay tối thiểu để từ multiset có `odd` ký tự lẻ đạt `odd ≤ 1` là `⌊odd/2⌋`. Mỗi lần thay đổi chẵn lẻ của **đúng hai** chữ (chữ bị thay và chữ thay vào) hoặc không đổi gì, nên `odd` giảm tối đa 2 mỗi lần → cần ít nhất `⌈(odd − 1)/2⌉ = ⌊odd/2⌋` lần. Và đạt được: mỗi lần thay một ký tự lẻ bằng một ký tự lẻ khác.

**Bổ đề 3 (XOR tiền tố).** Bit `c` của `mask[i]` là `(số lần c trong s[:i]) mod 2`. XOR hai mask = cộng mod 2 = hiệu mod 2 ⇒ bit `c` của `mask[r+1] ^ mask[l]` là chẵn lẻ của số lần `c` trong `s[l..r]`. Số bit 1 = `odd`.

Gộp lại: trả `True` ⟺ `odd // 2 ≤ k` ⟺ có thể thành palindrome.

## 11. Time / Space Complexity

Gọi `n = len(s)`, `Q = len(queries)`, `Σ = 26`.

| Cách | Tiền xử lý | Mỗi truy vấn | Space |
|------|-----------|--------------|-------|
| 1. Brute | — | `O(r − l + 1)` | `O(Σ)` |
| 2. Tiền tố 26 cột | `O(Σn)` | `O(Σ)` | `O(Σn)` |
| 3. Tiền tố chẵn lẻ | `O(Σn)` | `O(Σ)` | `O(Σn)` |
| 4. Bitmask XOR | `O(n)` | `O(1)` | `O(n)` |
| 5. Bitmask + cắt sớm | `O(n)` | `O(1)` | `O(n)` |

Tổng cách 4: `O(n + Q)`.

## 12. Bộ test

### 12.1 Ví dụ + edge case

In [16]:
impls = {
    "brute": can_make_brute,
    "prefix_counts": can_make_prefix_counts,
    "prefix_parity": can_make_prefix_parity,
    "bitmask": can_make_bitmask,
    "bitmask_early": can_make_bitmask_early,
    "Solution": Solution().canMakePaliQueries,
}
edge = [
    ("a", [[0, 0, 0]], [True]),
    ("ab", [[0, 1, 0], [0, 1, 1]], [False, True]),
    ("aaa", [[0, 2, 0]], [True]),
    ("abcdefghijklmnopqrstuvwxyz", [[0, 25, 12], [0, 25, 13]], [False, True]),
    ("aabbc", [[0, 4, 0], [1, 4, 0], [1, 3, 0]], [True, False, True]),   # "abbc" có a, c lẻ → cần 1 lần thay
]
for name, f in impls.items():
    for s, q, want in examples + edge:
        assert f(s, q) == want, (name, s, q)
print("all impls pass", len(examples + edge))

all impls pass 7


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

In [17]:
rng = random.Random(1177)
for _ in range(300):
    n = rng.randint(1, 15)
    s = "".join(rng.choice("abcd") for _ in range(n))
    qs = []
    for _ in range(rng.randint(1, 10)):
        l = rng.randint(0, n - 1); r = rng.randint(l, n - 1)
        qs.append([l, r, rng.randint(0, 4)])
    want = can_make_brute(s, qs)
    for name, f in impls.items():
        assert f(s, qs) == want, (name, s, qs)
print("random cross-check OK")

random cross-check OK


Đối chiếu thêm với brute force **thử mọi cách thay thật sự** (`can_brute_force_tiny`) trên đoạn rất ngắn — kiểm chứng chính công thức `odd // 2 <= k`, không chỉ cài đặt:

In [18]:
rng = random.Random(7)
for _ in range(150):
    t = "".join(rng.choice("abc") for _ in range(rng.randint(1, 5)))
    k = rng.randint(0, 3)
    assert can_brute_force_tiny(t, k, "abc") == Solution().canMakePaliQueries(t, [[0, len(t) - 1, k]])[0], (t, k)
print("công thức khớp brute force thực sự")

công thức khớp brute force thực sự


### 12.3 Test giới hạn

`n = Q = 10^5`. Cách 2/3 tốn nhiều bộ nhớ nên chỉ chạy cách bitmask.

In [19]:
rng = random.Random(10)
n = Q = 10**5
s_big = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))
q_big = []
for _ in range(Q):
    l = rng.randint(0, n - 1); r = rng.randint(l, min(n - 1, l + rng.randint(0, 60)))
    q_big.append([l, r, rng.randint(0, 15)])
for name in ["bitmask", "bitmask_early", "Solution"]:
    t0 = time.perf_counter()
    res = impls[name](s_big, q_big)
    print(f"{name:14} true={sum(res)} {time.perf_counter() - t0:.3f}s")
assert impls["bitmask"](s_big, q_big) == Solution().canMakePaliQueries(s_big, q_big)
assert can_make_brute(s_big, q_big[:2000]) == Solution().canMakePaliQueries(s_big, q_big[:2000])

bitmask        true=78299 0.047s
bitmask_early  true=78299 0.030s
Solution       true=78299 0.063s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — So sánh `odd <= k` (quên mỗi lần thay sửa được 2)

In [20]:
def wrong_odd_le_k(s, queries):
    return [sum(v % 2 for v in Counter(s[l:r + 1]).values()) <= k for l, r, k in queries]

s, q = "abc", [[0, 2, 1]]
print("sai:", wrong_odd_le_k(s, q), "| đúng:", can_make_brute(s, q))
assert wrong_odd_le_k("abc", [[0, 2, 1]]) != can_make_brute("abc", [[0, 2, 1]])

sai: [False] | đúng: [True]


**Vì sao sai:** `"abc"` có `odd = 3`, thay `c → a` được `"aba"` chỉ với **1** lần; `odd <= k` đòi `3 <= 1`. **Sửa:** `odd // 2 <= k`.

### Lỗi 2 — Dùng `(odd + 1) // 2` hoặc bắt `odd == 0` cho đoạn chẵn

In [21]:
def wrong_ceil(s, queries):
    return [(sum(v % 2 for v in Counter(s[l:r + 1]).values()) + 1) // 2 <= k for l, r, k in queries]

s, q = "aab", [[0, 2, 0]]
print("sai:", wrong_ceil(s, q), "| đúng:", can_make_brute(s, q))
assert wrong_ceil(s, q) != can_make_brute(s, q)

sai: [False] | đúng: [True]


**Vì sao sai:** đoạn độ dài lẻ được phép một ký tự lẻ ở giữa (`"aba"`). `(1 + 1) // 2 = 1 > 0` báo sai. **Sửa:** `odd // 2` — tự động cho phép một ký tự lẻ (và với đoạn chẵn, `odd` luôn chẵn nên `odd // 2` cũng đúng).

### Lỗi 3 — Lệch chỉ số tiền tố (`mask[r]` thay vì `mask[r + 1]`)

In [22]:
def wrong_index(s, queries):
    mask = [0] * (len(s) + 1)
    for i, ch in enumerate(s):
        mask[i + 1] = mask[i] ^ (1 << (ord(ch) - 97))
    return [bin(mask[r] ^ mask[l]).count("1") // 2 <= k for l, r, k in queries]

s, q = examples[0][:2]
print("sai:", wrong_index(s, q), "| đúng:", examples[0][2])
assert wrong_index(s, q) != examples[0][2]

sai: [True, True, True, True, False] | đúng: [True, False, False, True, True]


**Vì sao sai:** `mask[i]` là chẵn lẻ của `s[:i]` (không gồm `s[i]`). Đoạn `s[l..r]` (gồm `r`) cần `mask[r + 1]`. **Sửa:** `mask[r + 1] ^ mask[l]`.

### Lỗi 4 — Dùng `+`/`-` thay vì XOR cho mask

In [23]:
def wrong_minus(s, queries):
    mask = [0] * (len(s) + 1)
    for i, ch in enumerate(s):
        mask[i + 1] = mask[i] ^ (1 << (ord(ch) - 97))
    return [bin(abs(mask[r + 1] - mask[l])).count("1") // 2 <= k for l, r, k in queries]

# Tìm phản ví dụ tự động bằng ngẫu nhiên (seed cố định)
rng = random.Random(1)
found = None
for _ in range(2000):
    t = "".join(rng.choice("abc") for _ in range(6))
    l = rng.randint(0, 5); r = rng.randint(l, 5)
    if wrong_minus(t, [[l, r, 0]]) != can_make_brute(t, [[l, r, 0]]):
        found = (t, [l, r, 0]); break
print("phản ví dụ tìm được:", found)
assert found is not None

phản ví dụ tìm được: ('babcab', [4, 5, 0])


**Vì sao sai:** phép trừ có **mượn bit** (borrow) nên không tương ứng với "hiệu chẵn lẻ từng chữ". Chỉ XOR mới là phép cộng/trừ mod 2 độc lập trên từng bit. **Sửa:** `mask[r + 1] ^ mask[l]`.

## 14. Mẹo & nhận dạng pattern

- **"Hoán vị thành palindrome"** → chỉ quan tâm số ký tự có tần suất lẻ.
- **Chỉ cần chẵn lẻ của tần suất** trên bảng chữ nhỏ → **bitmask**, cập nhật bằng XOR.
- **Nhiều truy vấn trên đoạn** → tổng tiền tố; với chẵn lẻ, tiền tố XOR.
- **`10^5` truy vấn** → mỗi truy vấn phải `O(1)` hoặc `O(log n)` / `O(26)`.
- Họ bài **"mask tiền tố"**: LC 1371 (nguyên âm xuất hiện chẵn lần), LC 1542 (đoạn dài nhất là "awesome"), LC 1915 (số đoạn "wonderful").

## 15. Biến thể

1. **LC 1371 Find the Longest Substring Containing Vowels in Even Counts** — mask 5 bit, đoạn dài nhất có `mask[l] == mask[r+1]`.
2. **LC 1542 Find Longest Awesome Substring** — đoạn dài nhất hoán vị được thành palindrome: mask trùng hoặc lệch đúng 1 bit.
3. **LC 1915 Number of Wonderful Substrings** — đếm đoạn có tối đa một chữ lẻ: đếm mask tiền tố.
4. **LC 266 Palindrome Permutation** — một chuỗi duy nhất, `k = 0`.
5. Truy vấn trả về **số lần thay tối thiểu** thay vì true/false: chính là `odd // 2`.

In [24]:
# Biến thể 3: LC 1915 — đếm đoạn con có tối đa một chữ cái xuất hiện lẻ lần (chữ a..j)
def wonderful_substrings(word):
    cnt = Counter({0: 1})
    mask = ans = 0
    for ch in word:
        mask ^= 1 << (ord(ch) - 97)
        ans += cnt[mask]                         # đoạn với mọi chữ chẵn
        for b in range(10):
            ans += cnt[mask ^ (1 << b)]          # đoạn có đúng chữ b lẻ
        cnt[mask] += 1
    return ans

print(wonderful_substrings("aba"), wonderful_substrings("aabb"), wonderful_substrings("he"))
assert (wonderful_substrings("aba"), wonderful_substrings("aabb"), wonderful_substrings("he")) == (4, 9, 2)

4 9 2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Brute | Counter mỗi đoạn | `O(nQ)` | `O(Σ)` | Kiểm chứng |
| 2. Tiền tố 26 cột | Hiệu đếm | `O(Σ(n+Q))` | `O(Σn)` | Cần số lần chính xác |
| 3. Tiền tố chẵn lẻ | So sánh bit 0/1 | `O(Σ(n+Q))` | `O(Σn)` | Bước đệm sang bitmask |
| 4. Bitmask XOR | XOR + popcount | `O(n + Q)` | `O(n)` | **Nộp bài** |
| 5. Bitmask + cắt sớm | Thêm cận `k ≥ 13` | `O(n + Q)` | `O(n)` | Tối ưu hằng số |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `canMakePaliQueries("aabbccd", [[0,6,0],[0,5,0],[1,5,0],[1,5,1]])` = ?

**Bài 2 (tính tay):** liệt kê `mask[i]` (dạng nhị phân 3 bit c..a) cho `s = "cabac"`.

**Bài 3 (điền TODO):** viết `popcount(x)` không dùng `bin`/`bit_count` (mẹo `x &= x - 1`).

**Bài 4 (tự viết lại):** viết hàm trả về **số lần thay tối thiểu** cho mỗi truy vấn `[l, r]`.

**Bài 5 (mini challenge):** LC 1542 — độ dài đoạn con dài nhất của chuỗi chữ số `s` có thể hoán vị thành palindrome.

In [25]:
# Bài 3 — TODO
def popcount(x):
    # TODO: mỗi lần x &= x - 1 xoá bit 1 thấp nhất
    raise NotImplementedError

# assert popcount(0b1011) == 3 and popcount(0) == 0

In [26]:
# Bài 5 — TODO
def longest_awesome(s):
    # TODO: first[mask] = vị trí đầu tiên; xét mask trùng và lệch 1 bit (10 chữ số)
    pass

# assert longest_awesome("3242415") == 5
# assert longest_awesome("213123") == 6

## 18. Gợi ý 3 mức

- **Bài 1:** Mức 1: `"aabbccd"` odd = 1. Mức 2: `"aabbcc"` odd = 0; `"abbcc"` odd = 1. Mức 3: tất cả `True`.
- **Bài 2:** Mức 1: bắt đầu `000`. Mức 2: lật bit theo `c, a, b, a, c`. Mức 3: `000, 100, 101, 111, 110, 010`.
- **Bài 3:** Mức 1: đếm `c = 0`. Mức 2: `while x: x &= x - 1; c += 1`. Mức 3: số vòng = số bit 1.
- **Bài 4:** Mức 1: tính `odd` như cách 4. Mức 2: trả `odd // 2`. Mức 3: không cần `k`.
- **Bài 5:** Mức 1: `first = {0: -1}`. Mức 2: với mask hiện tại, xét `mask` và `mask ^ (1 << d)` cho `d` trong 0..9. Mức 3: chỉ lưu `first[mask]` lần đầu.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().canMakePaliQueries("aabbccd", [[0, 6, 0], [0, 5, 0], [1, 5, 0], [1, 5, 1]]) == [True, True, True, True]

# Bài 2
m, masks = 0, [0]
for ch in "cabac":
    m ^= 1 << (ord(ch) - 97)
    masks.append(m)
assert [f"{x:03b}" for x in masks] == ["000", "100", "101", "111", "110", "010"]

# Bài 3
def popcount_sol(x):
    c = 0
    while x:
        x &= x - 1
        c += 1
    return c

assert all(popcount_sol(x) == bin(x).count("1") for x in range(2000))

# Bài 4
def min_replacements(s, queries):
    mask = [0] * (len(s) + 1)
    for i, ch in enumerate(s):
        mask[i + 1] = mask[i] ^ (1 << (ord(ch) - 97))
    return [popcount_sol(mask[r + 1] ^ mask[l]) // 2 for l, r in queries]

assert min_replacements("abcda", [[0, 3], [0, 4], [1, 2]]) == [2, 1, 1]
print("bài 1-4 OK")

bài 1-4 OK


In [28]:
# Bài 5
def longest_awesome_sol(s):
    first = {0: -1}
    mask = best = 0
    for i, ch in enumerate(s):
        mask ^= 1 << int(ch)
        if mask in first:
            best = max(best, i - first[mask])
        for d in range(10):
            m2 = mask ^ (1 << d)
            if m2 in first:
                best = max(best, i - first[m2])
        first.setdefault(mask, i)
    return best

assert longest_awesome_sol("3242415") == 5
assert longest_awesome_sol("213123") == 6
assert longest_awesome_sol("12345678") == 1
rng = random.Random(15)
for _ in range(200):
    t = "".join(rng.choice("0123") for _ in range(rng.randint(1, 10)))
    brute = max(j - i for i in range(len(t)) for j in range(i + 1, len(t) + 1) if odd_count(t[i:j]) <= 1)
    assert longest_awesome_sol(t) == brute
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
mask = [0] * (len(s) + 1)
for i, ch in enumerate(s):
    mask[i + 1] = mask[i] ^ (1 << (ord(ch) - 97))
return [bin(mask[r + 1] ^ mask[l]).count('1') // 2 <= k for l, r, k in queries]
```

**Checklist:**

- [ ] Điều kiện `odd // 2 <= k` (không phải `odd <= k`)?
- [ ] Dùng `mask[r + 1] ^ mask[l]` (đoạn gồm cả `r`)?
- [ ] Dùng **XOR**, không dùng trừ?
- [ ] Mỗi truy vấn `O(1)`, tổng `O(n + Q)` cho `10^5`?
- [ ] Không sửa `s` giữa các truy vấn?